# GridSmart, Stage 2a: Weather Producer

**Replaying historical weather into Kafka as a simulated live sensor feed.**

This notebook stands in for a field weather station. It is written in **plain
Python with no Spark**, deliberately: a real weather station is a small
embedded controller with no distributed compute, so using Spark here would
model the problem incorrectly and hide any back-pressure the real system
would experience.

### The replay contract

Every **5 seconds**, emit **120 records**: five calendar days of hourly
readings, 24 per day.

Each day's 24-row block is stamped with a `weather_ts` one second apart:
day 1 at `t`, day 2 at `t+1`, and so on. That compresses five days of event
time into the five seconds of wall clock the batch occupies, so a "day" of
simulated grid activity elapses every second.

Downstream, the watermark and the windowed aggregations run against that
event time. The watermark is pinned to the same 5-second cadence, and the
coupling is asserted by a test so the two cannot drift apart.
→ [ADR 0005](../docs/adr/0005-watermark-and-late-data-policy.md)

A file pointer advances across ticks so the replay stays chronological and
never re-sends a record, wrapping back to the start at end of file.

**Run this first**, then `03_spark_streaming.ipynb`, then
`04_consumer_dashboard.ipynb`, all three run concurrently.

In [ ]:
# Shared project modules, imported, not redefined. The streaming job and the
# batch trainer use the same feature contract and schemas by construction.
import sys

sys.path.insert(0, "../src")

from gridsmart import config, features, producer, schemas, session, streaming

print(f"Kafka broker : {config.KAFKA_BOOTSTRAP}")
print(f"Topic in     : {config.TOPIC_WEATHER_IN}")
print(f"Cadence      : {config.BATCH_SIZE} records every {config.TICK_SECONDS}s")
print(f"Watermark    : {config.WATERMARK_DELAY} on event time")

In [1]:
# -----------------------------------------------------------
# Load weather dataset and prepare constants
# -----------------------------------------------------------

import pandas as pd

# --- File path and streaming configuration constants ---
WEATHER_CSV   = "dataset/weather.csv"   # path to weather data
ROWS_PER_DAY  = 24                      # each day has 24 hourly readings
DAYS_PER_TICK = 5                       # each streaming tick sends 5 days of data
BATCH_SIZE    = ROWS_PER_DAY * DAYS_PER_TICK  # total rows per tick = 24 × 5 = 120
TICK_SECONDS  = 5                       # send a new batch every 5 seconds

# --- Load and sort dataset chronologically (oldest first) ---
weather5s = pd.read_csv(WEATHER_CSV, parse_dates=["timestamp"])
weather5s = weather5s.sort_values(["site_id", "timestamp"]).reset_index(drop=True)

# --- Compute and display summary information ---
total_days = len(weather5s) // ROWS_PER_DAY

print(f"Loaded {len(weather5s)} weather records (~{total_days} days).")
print(f"Each tick will send {DAYS_PER_TICK} days ({BATCH_SIZE} rows) in chronological order.\n")

Loaded 11904 weather records (~496 days).
Each tick will send 5 days (120 rows) in chronological order.



This section loads and prepares the weather dataset for Kafka streaming. The file `dataset/weather.csv` contains hourly readings, so I set `ROWS_PER_DAY = 24` and configured `DAYS_PER_TICK = 5` to simulate five days of sensor data per streaming cycle. Each tick therefore sends `24 × 5 = 120` rows every 5 seconds in chronological order. The dataset is parsed with timestamps and sorted by `site_id` and `timestamp` to ensure realistic, time-ordered streaming. The print statement confirms this setup by displaying how many days and rows are sent per tick before producing data to Kafka.

In [2]:
# --------------------------------
# Set up Kafka producer
# --------------------------------

import json
from kafka import KafkaProducer

# Define Kafka connection and topic details
KAFKA_BOOTSTRAP = "kafka:9092"   # Kafka broker address
TOPIC = "weather5s"              # Kafka topic for streaming 5-second weather data

# Initialize the Kafka producer
producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BOOTSTRAP],       # Kafka server(s) to connect to
    value_serializer=lambda x: json.dumps(x).encode("utf-8"),  # Convert Python dicts to JSON and encode as bytes
    api_version=(0, 10),                       # Ensure compatibility with Kafka version 0.10+
)

print("Kafka Producer successfully initialized and ready to stream data.")

Kafka Producer successfully initialized and ready to stream data.


This section sets up the Apache Kafka producer that streams weather data in real time. I first import the required libraries (`json` and `KafkaProducer`) and define the Kafka broker address (`kafka:9092`) along with the topic name `weather5s`, which represents the 5-second weather stream. The producer is then configured with UTF-8 JSON serialization so that each batch of weather records is correctly encoded before transmission. The `api_version` ensures compatibility with the cluster version, allowing reliable and consistent message delivery to the Kafka topic.

In [ ]:
# ------------------------------------------
# Stream 5 days (120 rows) every 5 seconds
# ------------------------------------------
import time
import pandas as pd

pointer = 0  # Tracks the current position in the dataset to maintain order across iterations

while True:
    start = pointer                      # Define batch start index
    end = start + BATCH_SIZE              # Define batch end index (120 rows per tick)

    # Restart from top when reaching EOF
    if end > len(weather5s):
        print("\n[INFO] End of dataset reached, restarting from top.\n")
        pointer = 0
        start = 0
        end = BATCH_SIZE
        
    # Extract the next batch of 5 days (120 hourly records)
    batch = weather5s.iloc[start:end].copy()

    # Capture the current Unix timestamp as the base epoch for this batch
    base_epoch = int(time.time())

    # Assign weather_ts per 24-row block (Day 1..5 as base, base+1, ... base+4)
    rel_idx = batch.index - start
    day_block = (rel_idx // ROWS_PER_DAY).astype(int)
    batch["weather_ts"] = base_epoch + day_block

    # Send each row in the batch to Kafka
    for _, r in batch.iterrows():
        payload = {
            "site_id": int(r["site_id"]),
            "timestamp": r["timestamp"].isoformat(),
            "air_temperature": None if pd.isna(r.get("air_temperature")) else float(r["air_temperature"]),
            "cloud_coverage": None if pd.isna(r.get("cloud_coverage")) else int(r["cloud_coverage"]),
            "dew_temperature": None if pd.isna(r.get("dew_temperature")) else float(r["dew_temperature"]),
            "sea_level_pressure": None if pd.isna(r.get("sea_level_pressure")) else float(r["sea_level_pressure"]),
            "wind_direction": None if pd.isna(r.get("wind_direction")) else int(r["wind_direction"]),
            "wind_speed": None if pd.isna(r.get("wind_speed")) else float(r["wind_speed"]),
            "weather_ts": int(r["weather_ts"]),
        }
        producer.send(TOPIC, payload)    # Send encoded record to Kafka topic

    # Ensure all messages are delivered before next batch
    producer.flush()
    pointer = end  # move forward exactly 120 rows (Advance pointer for the next 5-day batch)

    # Print informative logs for monitoring progress
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(base_epoch))}] Batch sent: -> (ts = {base_epoch})")
    for day in range(DAYS_PER_TICK):
        day_start = start + day * ROWS_PER_DAY + 1
        day_end = start + (day + 1) * ROWS_PER_DAY
        ts_val = base_epoch + day
        print(f" Day {day+1} (records {day_start}-{day_end}), ts = {ts_val}")

    # Summary log for batch completion
    print(f"[INFO] Sent {BATCH_SIZE} records (rows {start+1}-{end}) | Next batch in {TICK_SECONDS}s...\n")

    # Wait before next batch
    time.sleep(TICK_SECONDS)


[2025-11-02 12:50:34] Batch sent: -> (ts = 1762087834)
  Day 1 (records 1-24) | ts = 1762087834
  Day 2 (records 25-48) | ts = 1762087835
  Day 3 (records 49-72) | ts = 1762087836
  Day 4 (records 73-96) | ts = 1762087837
  Day 5 (records 97-120) | ts = 1762087838
[INFO] Sent 120 records (rows 1-120) | Next batch in 5s...


[2025-11-02 12:50:39] Batch sent: -> (ts = 1762087839)
  Day 1 (records 121-144) | ts = 1762087839
  Day 2 (records 145-168) | ts = 1762087840
  Day 3 (records 169-192) | ts = 1762087841
  Day 4 (records 193-216) | ts = 1762087842
  Day 5 (records 217-240) | ts = 1762087843
[INFO] Sent 120 records (rows 121-240) | Next batch in 5s...



This section implements the live streaming loop for the Kafka producer. Every 5 seconds, it sends a batch of 120 weather records (representing 5 days × 24 hourly readings) to the Kafka topic `weather5s`. The script maintains a pointer to keep track of its current position in the dataset and automatically restarts from the top once it reaches the end. Each batch is assigned a base Unix timestamp (`weather_ts`) to simulate real-time data collection across five consecutive days. The loop prints detailed progress logs for every batch, including record ranges, timestamps, and upcoming batch intervals, ensuring full transparency and traceability of the data streaming process.

---

The producer runs until interrupted. Leave it running and start
`03_spark_streaming.ipynb` next.